In [1]:
import requests

def address_to_geolocation(address_details):
    """
    Given string representing desired address get back information about longitude, latitude

    Parameters:
    address_details (str): Detailed address information of desired location.

    Returns:
    dict: Dict that contains keys for longitude and latitude.
    """
    try:
        url = "http://api.positionstack.com/v1/forward"
        
        # Parameters including the API key and the query address
        params = {
            "access_key": "cb45718a97a93215b04a33aeadd75742",
            "query": address_details
        }
        
        # Sending the GET request to the API
        response = requests.get(url, params=params)
        
        # Checking if the request was successful
        if response.status_code == 200:
            data = response.json()
            if 'data' in data and data['data']:
                lat = data['data'][0]['latitude']
                lon = data['data'][0]['longitude']
                return {'latitude': lat, 'longitude': lon}
            else:
                raise ValueError("No data found for the given address.")
        else:
            # If the status code is not 200, raise an exception with a detailed error message
            response.raise_for_status()
    except requests.RequestException as e:
        # This captures exceptions raised by requests, including HTTPError, Timeout, etc.
        raise Exception(f"API request failed: {e}")
    except KeyError as e:
        # This captures errors like missing 'data' or 'latitude'/'longitude' keys in the response
        raise Exception(f"Data parsing error: Missing key {e}")


In [3]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": lat,
	"longitude": lon,
	"current": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "is_day", "precipitation", "rain", "cloud_cover", "pressure_msl", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m"],
	"timezone": "Europe/Berlin",
	"forecast_days": 1
}
responses = requests.get(url, params=params)

data = responses.json()['current']
desc = responses.json()['current_units']
desc['is_day'] = '1 for day else night'
desc = {k + '_info': v for k, v in desc.items()}
combined = {k: v for pair in zip(data.items(), desc.items()) for k, v in [(pair[0][0], pair[0][1]), (pair[0][0] + '_info', pair[1][1])]}

NameError: name 'lat' is not defined

In [7]:
def get_weather_forecast(lat, lon):
    """
    Retrieves weather forecast information for a given latitude and longitude.

    Parameters:
    lat (float): Latitude of the location.
    lon (float): Longitude of the location.

    Returns:
    dict: Dictionary containing weather data and descriptive information for each metric.
    """
    try:
        # Define the API URL and parameters
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": lat,
            "longitude": lon,
            "current": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "is_day", "precipitation", "rain", "cloud_cover", "pressure_msl", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m"],
            "timezone": "Europe/Berlin",
            "forecast_days": 1
        }
        
        # Send the request to the weather API
        response = requests.get(url, params=params)
        
        # Check if the request was successful
        if response.status_code == 200:
            # Extract weather data and descriptions from the response
            data = response.json()['current']
            desc = response.json()['current_units']
            
            # Modify descriptions as needed
            desc['is_day'] = '1 for day, 0 for night'
            desc = {k + '_info': v for k, v in desc.items()}
            
            # Combine data with descriptions
            combined = {k: v for pair in zip(data.items(), desc.items()) for k, v in [(pair[0][0], pair[0][1]), (pair[0][0] + '_info', pair[1][1])]}
            
            return combined
        else:
            # Raise an error if the API call was unsuccessful
            response.raise_for_status()
    
    except requests.RequestException as e:
        raise Exception(f"API request failed: {e}")
    except KeyError as e:
        raise Exception(f"Data parsing error: Missing key {e}")

In [4]:
info = address_to_geolocation('Wąwozowa 32b, Kraków')

In [8]:
get_weather_forecast(info['latitude'], info['longitude'])

{'time': '2024-10-26T17:00',
 'time_info': 'iso8601',
 'interval': 900,
 'interval_info': 'seconds',
 'temperature_2m': 14.2,
 'temperature_2m_info': '°C',
 'relative_humidity_2m': 74,
 'relative_humidity_2m_info': '%',
 'apparent_temperature': 12.4,
 'apparent_temperature_info': '°C',
 'is_day': 1,
 'is_day_info': '1 for day, 0 for night',
 'precipitation': 0.0,
 'precipitation_info': 'mm',
 'rain': 0.0,
 'rain_info': 'mm',
 'cloud_cover': 92,
 'cloud_cover_info': '%',
 'pressure_msl': 1023.3,
 'pressure_msl_info': 'hPa',
 'wind_speed_10m': 11.9,
 'wind_speed_10m_info': 'km/h',
 'wind_direction_10m': 70,
 'wind_direction_10m_info': '°',
 'wind_gusts_10m': 20.2,
 'wind_gusts_10m_info': 'km/h'}

In [ ]:

import os
from dotenv import load_dotenv
from openai import AzureOpenAI
from prompts.prompts import synthesis_system, synthesis_user
load_dotenv()


gpt_name = 'GPT4_O_' 
model_name = 'gpt-4o-ArturG'

api_key = os.getenv(gpt_name + "AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv(gpt_name + "AZURE_OPENAI_ENDPOINT")
api_version = os.getenv(gpt_name + "OPENAI_API_VERSION")

# Initialize the AzureOpenAI client
client = AzureOpenAI(
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version
)


# Prepare messages and request parameters
messages = [{"role": "system", "content": 'You are bor_dude telling best bro jokes ever '},
            {"role": "user", "content": 'Tell me best bro joke ever '}]
request_params = {
    "model": model_name,
    "messages": messages
}

response = client.chat.completions.create(**request_params)
answer = response.choices[0].message.content
answer

In [6]:
response.usage.completion_tokens, response.usage.prompt_tokens

(22, 29)

In [ ]:
[['27-10-2024 12:30', {'api': "Convert the address 'Wawozowa 32, Krakow, Poland' to geolocation coordinates"}, 40, 886],
 ['27-10-2024 12:30', {'api': "Convert the address 'Wawozowa 32, Krakow, Poland' to geolocation coordinates"}, 40, 886,
 ['27-10-2024 12:30', {'tool_choice': 'address_to_geolocation', 'tool_input': 'Wawozowa 32, Krakow, Poland'}, 31, 666]]